In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/titanic.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    str    
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    str    
 8   class        891 non-null    str    
 9   who          891 non-null    str    
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    str    
 12  embark_town  889 non-null    str    
 13  alive        891 non-null    str    
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), str(7)
memory usage: 92.4 KB


In [2]:
df["survived"] = df["survived"].astype(bool)
df["pclass"] = df["pclass"].astype("category")

In [3]:
categorical_cols = ["sex", "embarked", "class", "who", "embark_town", "alive"]
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

In [4]:
# age: ~19.87% missing — impute with median, flag imputed rows
df["age_missing"] = df["age"].isna()
df["age"] = df["age"].fillna(df["age"].median())

# deck: ~77.22% missing — too sparse to be useful, drop column
df = df.drop(columns=["deck"])

# embarked/embark_town: 0.22% missing (2 rows) — drop those rows
df = df.dropna(subset=["embarked", "embark_town"])

In [5]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows ({before} → {after})")

Removed 111 duplicate rows (889 → 778)


In [6]:
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["is_alone"] = df["family_size"] == 1
df["fare_per_person"] = df["fare"] / df["family_size"]

In [7]:
df.to_csv("data/titanic_cleaned.csv", index=False)
print(df.shape)
df.head()

(778, 18)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,age_missing,family_size,is_alone,fare_per_person
0,False,3,male,22.0,1,0,7.2500,s,third,man,True,southampton,no,False,False,2,False,3.62500
1,True,1,female,38.0,1,0,71.2833,c,first,woman,False,cherbourg,yes,False,False,2,False,35.64165
2,True,3,female,26.0,0,0,7.9250,s,third,woman,False,southampton,yes,True,False,1,True,7.92500
3,True,1,female,35.0,1,0,53.1000,s,first,woman,False,southampton,yes,False,False,2,False,26.55000
4,False,3,male,35.0,0,0,8.0500,s,third,man,True,southampton,no,True,False,1,True,8.05000


## Transformation Summary
- Converted `survived` to bool, `pclass` to category
- Standardized text columns: stripped whitespace, lowercased
- `age`: 19.87% missing → imputed with median, flagged via `age_missing`
- `deck`: 77.22% missing → dropped, too sparse to be useful
- `embarked`/`embark_town`: 0.22% missing → dropped 2 affected rows
- Removed 107 fully duplicate rows
- Derived `family_size`, `is_alone`, `fare_per_person`
- Output saved to `data/titanic_cleaned.csv`; raw file untouched